# Hourly HSS catalogs — B1

**Method source:** `/Users/aosh/Developer/sdds/downloader/dbscripts/hss/modelB1.sql`; gap filling: `/Users/aosh/Developer/sdds/downloader/dbscripts/hss/ace_fill.sql`.

For the merged hourly parquets (`V` in km/s, `N_p` in cm⁻³, `T_p` in K), flag `V > 400`, `N_p < 6.1`, and `T_p > 82000`. Keep these flags and add hours with >12 raw positives in a centered 25-hour window. Retain events spanning ≥24 h, or ≥12 h with peak speed >450 km/s.

The SQL gap fill is preserved: interpolate missing values in groups of 2–25 rows with valid endpoints; otherwise substitute zero. Windows are evaluated chronologically. Isolated flags and the first/last grid samples are excluded as in SQL. Thresholds are applied unchanged to all satellites.

Outputs: `Outputs/HSS List/SAT_NAME_hss_b1.csv` with `T_start`, `T_end`, `T_max` (earliest peak; UTC), `v_max` (km/s), `n_mean` (cm⁻³), and `t_mean` (K). Means use the B1-filled values over the retained event hours, including any zero substitutions.

**ICME flags:** `icme_flag` is `overlap` when the HSS and ICME intervals intersect (including endpoints), `near` when they are separated by at most 12 hours on either side, and blank otherwise. Overlap takes precedence; no HSS events are removed.

**ICME sources:** `~/Developer/Data Links/SINP ICME List.csv` for ACE (`T_shock`, falling back to `T_start`, through `T_end`); `~/Developer/Data Links/HELIO4CAST ICME Catalog.csv` for STEREO-A/B, PSP, and Solar Orbiter (`icme_start_time` through `mo_end_time`, including sheath and magnetic obstacle). All times are UTC. Only complete, non-reversed intervals are matched; the linked ACE snapshot has 150 missing-end and 3 reversed intervals, which are omitted. Blank means no match in the usable catalog intervals, including outside catalog coverage; it does not establish ICME absence.

**Yermolaev/SRI RAS generator (last cell):** source intervals are `/Users/aosh/Developer/ICME_list/out/source/iki_sw_types.pkl`, produced by `/Users/aosh/Developer/ICME_list/downloader/iki_list.py`. Consecutive `SWfast`/`CIR` rows are joined only when the next start is one hour after the previous end (inclusive hourly labels); other types and gaps break events. CIR-only runs are retained. Earth/OMNI speed and temperature come from `/Users/aosh/Developer/ICME_list/out/source/omni_v_dst.pkl`. Peaks and temperature means use valid samples inside the unpadded event interval, without interpolation. Missing peaks remain blank; `n_mean` is blank because this OMNI cache has no density. Output: `Outputs/HSS List/yermolaev_hss.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
archive_dir = Path("/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged")
parquet_paths = sorted(archive_dir.glob("*.parquet"))
output_dir = Path("/Users/aosh/Developer/helio-n/Outputs/HSS List")
data_links = Path.home() / "Developer" / "Data Links"
merged_list_path = data_links / "SINP ICME List.csv"
icmecat_path = data_links / "HELIO4CAST ICME Catalog.csv"
near_tolerance = pd.Timedelta(hours=12)

In [3]:
def find_hss_b1(plasma):
    """Apply the chronological SDDS B1 algorithm to hourly plasma columns."""
    plasma = plasma.loc[:, ["speed", "density", "temperature"]].copy()
    plasma.index = pd.to_datetime(plasma.index, utc=True)
    plasma = plasma.sort_index().astype(float)
    plasma = plasma.reindex(pd.date_range(plasma.index.min(), plasma.index.max(), freq="h"))

    # Port ace_fill.sql's group definition and endpoint tests, including zero fill.
    filled = plasma.copy()
    positions = pd.Series(np.arange(len(plasma)), index=plasma.index)
    for column in plasma.columns:
        values = plasma[column].to_numpy()
        present = plasma[column].notna()
        groups = (present & present.shift(1, fill_value=False)).cumsum()
        grouped_positions = positions.groupby(groups)
        first = grouped_positions.transform("min").to_numpy()
        last = grouped_positions.transform("max").to_numpy()
        count = grouped_positions.transform("size").to_numpy()
        missing = np.isnan(values)
        interpolate = missing & (count > 1) & (count < 26) & np.isfinite(values[first]) & np.isfinite(values[last])
        result = values.copy()
        result[missing] = 0.0
        result[interpolate] = values[first[interpolate]] + (
            values[last[interpolate]] - values[first[interpolate]]
        ) * (positions.to_numpy()[interpolate] - first[interpolate]) / (last[interpolate] - first[interpolate])
        filled[column] = result

    raw = (filled.speed > 400) & (filled.density < 6.1) & (filled.temperature > 82000)
    marked = raw | (raw.rolling(25, center=True, min_periods=1).sum() > 12)
    groups = (~marked).cumsum()

    # Reproduce SQL's s/b/e labels: adjacent positives, with both neighbors defined.
    retained = marked & (marked.shift(1, fill_value=False) | marked.shift(-1, fill_value=False))
    retained.iloc[0] = False
    retained.iloc[-1] = False
    selected = filled.loc[retained].copy()
    selected["dt"] = selected.index
    selected["group"] = groups.loc[retained]
    events = selected.groupby("group", sort=False).agg(
        start_dt=("dt", "min"),
        end_dt=("dt", "max"),
        vmax_dt=("speed", "idxmax"),
        vmax=("speed", "max"),
        samples=("speed", "size"),
        n_mean=("density", "mean"),
        t_mean=("temperature", "mean"),
    )
    duration_h = (events.end_dt - events.start_dt).dt.total_seconds() / 3600
    keep = (events.samples >= 12) & (
        (duration_h >= 24) | ((duration_h >= 12) & (events.vmax > 450))
    )
    return events.loc[keep, ["start_dt", "end_dt", "vmax_dt", "vmax", "n_mean", "t_mean"]].rename(
        columns={"start_dt": "T_start", "end_dt": "T_end", "vmax_dt": "T_max", "vmax": "v_max"}
    ).reset_index(drop=True)

In [4]:
merged = pd.read_csv(merged_list_path, usecols=["T_shock", "T_start", "T_end"])
merged = merged.apply(pd.to_datetime, utc=True)
merged["T_start"] = merged.T_shock.fillna(merged.T_start)
icme_windows = {"ace": merged.loc[merged.T_end >= merged.T_start, ["T_start", "T_end"]]}

icmecat = pd.read_csv(icmecat_path, usecols=["sc_insitu", "icme_start_time", "mo_end_time"])
icmecat["icme_start_time"] = pd.to_datetime(icmecat.icme_start_time, utc=True)
icmecat["mo_end_time"] = pd.to_datetime(icmecat.mo_end_time, utc=True)
for satellite, spacecraft in {
    "stereo-a": "STEREO-A", "stereo-b": "STEREO-B", "psp": "PSP", "solo": "SolarOrbiter"
}.items():
    windows = icmecat.loc[icmecat.sc_insitu == spacecraft].rename(
        columns={"icme_start_time": "T_start", "mo_end_time": "T_end"}
    )
    icme_windows[satellite] = windows.loc[windows.T_end >= windows.T_start, ["T_start", "T_end"]]

pd.DataFrame({
    satellite: {"intervals": len(windows), "first_start": windows.T_start.min(), "last_end": windows.T_end.max()}
    for satellite, windows in icme_windows.items()
}).T

,intervals,first_start,last_end
ace,620,2010-01-01 22:00:00+00:00,2025-01-03 22:00:00+00:00
stereo-a,363,2007-01-14 12:11:00+00:00,2025-08-11 08:20:00+00:00
stereo-b,150,2007-01-14 12:24:00+00:00,2014-09-26 16:32:00+00:00
psp,145,2018-10-30 20:25:00+00:00,2025-04-26 09:57:00+00:00
solo,159,2020-04-19 05:06:00+00:00,2025-05-27 11:27:00+00:00


In [5]:
def flag_icmes(events, windows, tolerance):
    events["icme_flag"] = ""
    for start, end in windows.itertuples(index=False, name=None):
        near = (events.T_start <= end + tolerance) & (events.T_end >= start - tolerance)
        overlap = (events.T_start <= end) & (events.T_end >= start)
        events.loc[near & events.icme_flag.ne("overlap"), "icme_flag"] = "near"
        events.loc[overlap, "icme_flag"] = "overlap"
    return events

In [6]:
output_dir.mkdir(parents=True, exist_ok=True)
catalogs = {}
for parquet_path in parquet_paths:
    satellite = parquet_path.stem.split("_")[0]
    plasma = pd.read_parquet(parquet_path, columns=["V", "N_p", "T_p"]).rename(
        columns={"V": "speed", "N_p": "density", "T_p": "temperature"}
    )
    events = find_hss_b1(plasma)
    catalogs[satellite] = flag_icmes(events, icme_windows[satellite], near_tolerance)
    catalogs[satellite].to_csv(output_dir / f"{satellite}_hss_b1.csv", index=False)

pd.DataFrame({
    satellite: events.icme_flag.replace("", "empty").value_counts()
    for satellite, events in catalogs.items()
}).reindex(["overlap", "near", "empty"]).fillna(0).astype(int).T

icme_flag,overlap,near,empty
ace,137,76,459
psp,0,0,0
solo,9,7,89
stereo-a,60,35,508
stereo-b,29,9,115


In [7]:
hss_events = pd.concat(catalogs, names=["satellite", "event"])
hss_events

T_start                     T_end  \
satellite event                                                       
ace       0     2010-01-11 16:00:00+00:00 2010-01-12 04:00:00+00:00   
          1     2010-01-13 07:00:00+00:00 2010-01-14 17:00:00+00:00   
          2     2010-01-20 23:00:00+00:00 2010-01-21 22:00:00+00:00   
          3     2010-02-01 10:00:00+00:00 2010-02-03 17:00:00+00:00   
          4     2010-02-18 12:00:00+00:00 2010-02-19 14:00:00+00:00   
...                                   ...                       ...   
stereo-b  148   2014-08-27 21:00:00+00:00 2014-08-30 01:00:00+00:00   
          149   2014-09-02 16:00:00+00:00 2014-09-05 09:00:00+00:00   
          150   2014-09-07 11:00:00+00:00 2014-09-09 09:00:00+00:00   
          151   2014-09-14 00:00:00+00:00 2014-09-15 06:00:00+00:00   
          152   2014-09-16 13:00:00+00:00 2014-09-17 17:00:00+00:00   

                                    T_max       v_max    n_mean  \
satellite event                                                   
ace       0     2010-01-11 18:00:00+00:00  492.489990  4.282154   
          1     2010-01-14 01:00:00+00:00  513.219971  2.996731   
          2     2010-01-21 00:00:00+00:00  510.459991  2.398829   
          3     2010-02-03 02:00:00+00:00  557.679993  3.910695   
          4     2010-02-19 08:00:00+00:00  487.769989  2.175694   
...                                   ...         ...       ...   
stereo-b  148   2014-08-29 05:00:00+00:00  446.000000  4.544906   
          149   2014-09-03 10:00:00+00:00  772.599976  0.000000   
          150   2014-09-08 14:00:00+00:00  541.900024  0.818723   
          151   2014-09-14 02:00:00+00:00  526.299988  4.653226   
          152   2014-09-17 05:00:00+00:00  431.200012  4.463448   

                        t_mean icme_flag  
satellite event                           
ace       0      113638.539663            
          1      111189.519866   overlap  
          2       99346.729492            
          3      124570.371233            
          4      159654.981337      near  
...                        ...       ...  
stereo-b  148    101818.698113            
          149    192909.984848   overlap  
          150    164890.361702            
          151    121556.451613            
          152    107271.620690            

[1533 rows x 7 columns]

In [8]:
# Yermolaev/SRI RAS intervals and Earth/OMNI event summaries.
iki_path = Path("/Users/aosh/Developer/ICME_list/out/source/iki_sw_types.pkl")
omni_path = Path("/Users/aosh/Developer/ICME_list/out/source/omni_v_dst.pkl")
iki = pd.read_pickle(iki_path).sort_values("Start")
iki[["Start", "End"]] = iki[["Start", "End"]].apply(pd.to_datetime, utc=True)
is_hss = iki.Type.isin(["SWfast", "CIR"])
groups = (~is_hss | iki.Start.ne(iki.End.shift() + pd.Timedelta(hours=1))).cumsum()
yermolaev_hss = iki.loc[is_hss].groupby(groups[is_hss]).agg(
    T_start=("Start", "min"), T_end=("End", "max")
).reset_index(drop=True)

omni = pd.read_pickle(omni_path).set_index("date").sort_index()
omni.index = pd.to_datetime(omni.index, utc=True)
omni = omni[["v", "temperature"]].apply(pd.to_numeric)
intervals = [omni.loc[row.T_start:row.T_end] for row in yermolaev_hss.itertuples()]
yermolaev_hss["T_max"] = [data.v.idxmax() if data.v.notna().any() else pd.NaT for data in intervals]
yermolaev_hss["v_max"] = [data.v.max() for data in intervals]
yermolaev_hss["n_mean"] = np.nan
yermolaev_hss["t_mean"] = [data.temperature.mean() for data in intervals]
yermolaev_hss.to_csv(output_dir / "yermolaev_hss.csv", index=False)
yermolaev_hss

,T_start,T_end,T_max,v_max,n_mean,t_mean
0,1996-01-02 08:00:00+00:00,1996-01-03 14:00:00+00:00,1996-01-03 11:00:00+00:00,545.0,NaN,123280.903226
1,1996-01-03 18:00:00+00:00,1996-01-03 21:00:00+00:00,1996-01-03 18:00:00+00:00,455.0,NaN,84404.000000
2,1996-01-04 09:00:00+00:00,1996-01-04 13:00:00+00:00,1996-01-04 10:00:00+00:00,459.0,NaN,94423.400000
3,1996-01-04 22:00:00+00:00,1996-01-04 23:00:00+00:00,1996-01-04 23:00:00+00:00,459.0,NaN,127844.000000
4,1996-01-12 18:00:00+00:00,1996-01-13 01:00:00+00:00,1996-01-13 01:00:00+00:00,387.0,NaN,76829.625000
...,...,...,...,...,...,...
2776,2024-11-03 20:00:00+00:00,2024-11-04 02:00:00+00:00,2024-11-03 23:00:00+00:00,464.0,NaN,43938.000000
2777,2024-11-04 12:00:00+00:00,2024-11-04 12:00:00+00:00,2024-11-04 12:00:00+00:00,458.0,NaN,135279.000000
2778,2024-11-05 14:00:00+00:00,2024-11-05 16:00:00+00:00,2024-11-05 15:00:00+00:00,458.0,NaN,144354.666667
2779,2024-11-05 23:00:00+00:00,2024-11-06 10:00:00+00:00,2024-11-06 04:00:00+00:00,463.0,NaN,119197.416667
